In [11]:
# ===============================
# HOTEL DATA: FINANCIAL + STOCK PERFORMANCE (2018–2025)
# ===============================

import pandas as pd
import wrds

# --- 1. Connect to WRDS ---
print("Connecting to WRDS...")
db = wrds.Connection()
print("WRDS connection established.\n")

# --- 2. Define the 4 hotel companies by ticker ---
hotel_tickers = ['MAR', 'HLT', 'IHG', 'H']  # Hyatt changed to 'H', exclude Accor

# --- 3. Pull Compustat annual financial data ---
print("Fetching Compustat annual financial data (2018–2025)...")
comp_query = f"""
SELECT gvkey AS company_id,
       datadate AS fiscal_date,
       tic AS ticker,
       conm AS company_name,
       at AS total_assets,
       sale AS revenue,
       ni AS net_income,
       ceq AS shareholders_equity,
       cshpri AS shares_outstanding,
       prcc_f AS price_fiscal_year_end,
       oancf AS operating_cash_flow,
       capx AS capital_expenditure
FROM comp.funda
WHERE tic IN ({','.join([f"'{t}'" for t in hotel_tickers])})
AND datadate BETWEEN '2018-01-01' AND '2025-12-31'
AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
"""
comp_clean = db.raw_sql(comp_query)
comp_clean['fiscal_date'] = pd.to_datetime(comp_clean['fiscal_date'])
comp_clean['year'] = comp_clean['fiscal_date'].dt.year
print(f"Compustat data loaded: {comp_clean.shape[0]} rows\n")

# --- 4. Calculate financial ratios/metrics ---
print("Calculating financial ratios/metrics...")
comp_clean["profit_margin"] = comp_clean["net_income"] / comp_clean["revenue"]
comp_clean["asset_turnover"] = comp_clean["revenue"] / comp_clean["total_assets"]
comp_clean["ROA"] = comp_clean["net_income"] / comp_clean["total_assets"]
comp_clean["ROE"] = comp_clean["net_income"] / comp_clean["shareholders_equity"]
comp_clean["EPS"] = comp_clean["net_income"] / comp_clean["shares_outstanding"]
comp_clean["PE_ratio"] = comp_clean["price_fiscal_year_end"] / comp_clean["EPS"]
comp_clean["PB_ratio"] = comp_clean["price_fiscal_year_end"] / (comp_clean["shareholders_equity"] / comp_clean["shares_outstanding"])
comp_clean["FCF"] = comp_clean["operating_cash_flow"] - comp_clean["capital_expenditure"]

# Round numbers for readability
numeric_cols = ["ROA", "ROE", "PE_ratio", "PB_ratio", "FCF",
                "profit_margin", "asset_turnover", "EPS",
                "total_assets", "revenue", "net_income", "shareholders_equity"]
comp_clean[numeric_cols] = comp_clean[numeric_cols].round(2)
print("Financial ratios calculated and rounded.\n")

# --- 5. Pull CRSP monthly stock data for these hotels ---
print("Fetching CRSP monthly stock data (2018–2024)...")
gvkeys = comp_clean['company_id'].unique().tolist()
crsp_query = f"""
SELECT permno AS company_id_crsp,
       date AS trade_date,
       ret AS return,
       prc AS price,
       vol AS volume,
       shrout AS shares_outstanding_raw
FROM crsp.msf
WHERE permno IN (
    SELECT lpermno
    FROM crsp.ccmxpf_linktable
    WHERE gvkey IN ({','.join([f"'{g}'" for g in gvkeys])})
    AND linktype IN ('LU','LC') AND usedflag=1
)
AND date BETWEEN '2018-01-01' AND '2024-12-31'
"""
crsp_clean = db.raw_sql(crsp_query)
crsp_clean['trade_date'] = pd.to_datetime(crsp_clean['trade_date'])
crsp_clean['year'] = crsp_clean['trade_date'].dt.year
crsp_clean['shares_outstanding'] = crsp_clean['shares_outstanding_raw'] * 1000
crsp_clean['market_cap'] = crsp_clean['price'] * crsp_clean['shares_outstanding']
print(f"CRSP data loaded: {crsp_clean.shape[0]} rows\n")

# --- 6. Merge Compustat and CRSP data ---
print("Merging CRSP with Compustat via link table...")
link_table = db.raw_sql(f"""
SELECT lpermno AS permno, gvkey
FROM crsp.ccmxpf_linktable
WHERE gvkey IN ({','.join([f"'{g}'" for g in gvkeys])})
AND linktype IN ('LU','LC') AND usedflag=1
""")
merged_clean = crsp_clean.merge(link_table, left_on='company_id_crsp', right_on='permno', how='inner')
merged_clean = merged_clean.merge(comp_clean, left_on='gvkey', right_on='company_id', how='left', suffixes=('_crsp','_comp'))
merged_clean = merged_clean.reset_index(drop=True)
print(f"Final merged dataset: {merged_clean.shape[0]} rows\n")

# --- 7. Save final dataset ---
print("Saving final dataset to CSV and Parquet...")
merged_clean.to_csv("top4_hotels_2018_2025.csv", index=False)
merged_clean.to_parquet("top4_hotels_2018_2025.parquet", index=False)
print("Dataset saved successfully!\n")

# --- 8. Preview the data ---
print("Preview of merged dataset:")
merged_clean.head(10)

Connecting to WRDS...


Enter your WRDS username [work]: karinaermakova
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
WRDS connection established.

Fetching Compustat annual financial data (2018–2025)...
Compustat data loaded: 32 rows

Calculating financial ratios/metrics...
Financial ratios calculated and rounded.

Fetching CRSP monthly stock data (2018–2024)...
CRSP data loaded: 336 rows

Merging CRSP with Compustat via link table...
Final merged dataset: 3360 rows

Saving final dataset to CSV and Parquet...
Dataset saved successfully!

Preview of merged dataset:


,company_id_crsp,trade_date,return,price,volume,shares_outstanding_raw,year_crsp,shares_outstanding_crsp,market_cap,permno,...,capital_expenditure,year_comp,profit_margin,asset_turnover,ROA,ROE,EPS,PE_ratio,PB_ratio,FCF
0,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,72.0,2018,0.09,0.64,0.05,1.39,2.53,28.38,39.35,1183.0
1,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,81.0,2019,0.09,0.63,0.06,-1.83,3.07,36.13,-66.04,1303.0
2,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,46.0,2020,-0.17,0.26,-0.04,0.48,-2.58,-43.1,-20.68,662.0
3,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,35.0,2021,0.07,0.37,0.03,-0.5,1.47,106.15,-53.01,74.0
4,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,39.0,2022,0.14,0.57,0.08,-1.14,4.56,27.69,-31.53,1642.0
5,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,151.0,2023,0.11,0.66,0.07,-0.48,4.35,41.81,-20.22,1795.0
6,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,96.0,2024,0.14,0.68,0.09,-0.41,6.19,39.93,-16.45,1917.0
7,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,101.0,2025,0.12,0.72,0.09,-0.27,6.17,46.53,-12.58,2028.0
8,14338,2018-02-28,-0.056743,80.79,561565.0,316118.0,2018,316118000.0,25539173220.000004,14338.0,...,72.0,2018,0.09,0.64,0.05,1.39,2.53,28.38,39.35,1183.0
9,14338,2018-02-28,-0.056743,80.79,561565.0,316118.0,2018,316118000.0,25539173220.000004,14338.0,...,81.0,2019,0.09,0.63,0.06,-1.83,3.07,36.13,-66.04,1303.0


In [15]:
# --- Check available years in the dataset ---

# Financial data years
financial_years = sorted(merged_clean['fiscal_date'].dt.year.unique())
print("Financial data available for years:", financial_years)
print("Last financial year available:", financial_years[-1])

# Stock data years
stock_years = sorted(merged_clean['trade_date'].dt.year.unique())
print("Stock data available for years:", stock_years)
print("Last stock year available:", stock_years[-1])

# Optional: show which companies are included
companies = merged_clean[['company_name', 'ticker', 'company_id_crsp']].drop_duplicates()
print("\nCompanies in the dataset:")
print(companies)

Financial data available for years: [np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Last financial year available: 2025
Stock data available for years: [np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
Last stock year available: 2024

Companies in the dataset:
                     company_name ticker  company_id_crsp
0       HILTON WORLDWIDE HOLDINGS    HLT            14338
672             MARRIOTT INTL INC    MAR            85913
1008  INTERCONTINENTAL HOTELS GRP    IHG            89704
1184            HYATT HOTELS CORP      H            93098


In [16]:
# --- Check for duplicate rows ---
duplicate_count = merged_clean.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 672


In [17]:
# --- Remove duplicate rows ---
merged_clean = merged_clean.drop_duplicates().reset_index(drop=True)

# --- Quick check after removal ---
print("Rows, Columns after removing duplicates:", merged_clean.shape)

Rows, Columns after removing duplicates: (2688, 33)


In [18]:
# Check missing values in the dataset
missing_summary = merged_clean.isna().sum()
print("Missing values per column:\n", missing_summary)

# Optional: check only key columns for your app
key_columns = ["company_name", "ticker", "fiscal_date", "trade_date", 
               "price", "return", "market_cap",
               "ROA", "ROE", "PE_ratio", "PB_ratio", "profit_margin", "FCF"]
print("\nMissing values in key columns:\n", merged_clean[key_columns].isna().sum())

Missing values per column:
 company_id_crsp            0
trade_date                 0
return                     0
price                      0
volume                     0
shares_outstanding_raw     0
year_crsp                  0
shares_outstanding_crsp    0
market_cap                 0
permno                     0
gvkey                      0
company_id                 0
fiscal_date                0
ticker                     0
company_name               0
total_assets               0
revenue                    0
net_income                 0
shareholders_equity        0
shares_outstanding_comp    0
price_fiscal_year_end      0
operating_cash_flow        0
capital_expenditure        0
year_comp                  0
profit_margin              0
asset_turnover             0
ROA                        0
ROE                        0
EPS                        0
PE_ratio                   0
PB_ratio                   0
FCF                        0
year                       0
dtype: int64

M

In [19]:
# Check the size of the dataset
print("Rows, Columns:", merged_clean.shape)

# Check column names
print("Columns:", merged_clean.columns.tolist())

# Preview first 10 rows
print("Preview of first 10 rows:")
merged_clean.head(10)

Rows, Columns: (2688, 33)
Columns: ['company_id_crsp', 'trade_date', 'return', 'price', 'volume', 'shares_outstanding_raw', 'year_crsp', 'shares_outstanding_crsp', 'market_cap', 'permno', 'gvkey', 'company_id', 'fiscal_date', 'ticker', 'company_name', 'total_assets', 'revenue', 'net_income', 'shareholders_equity', 'shares_outstanding_comp', 'price_fiscal_year_end', 'operating_cash_flow', 'capital_expenditure', 'year_comp', 'profit_margin', 'asset_turnover', 'ROA', 'ROE', 'EPS', 'PE_ratio', 'PB_ratio', 'FCF', 'year']
Preview of first 10 rows:


,company_id_crsp,trade_date,return,price,volume,shares_outstanding_raw,year_crsp,shares_outstanding_crsp,market_cap,permno,...,year_comp,profit_margin,asset_turnover,ROA,ROE,EPS,PE_ratio,PB_ratio,FCF,year
0,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2018,0.09,0.64,0.05,1.39,2.53,28.38,39.35,1183.0,2018
1,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2019,0.09,0.63,0.06,-1.83,3.07,36.13,-66.04,1303.0,2018
2,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2020,-0.17,0.26,-0.04,0.48,-2.58,-43.1,-20.68,662.0,2018
3,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2021,0.07,0.37,0.03,-0.5,1.47,106.15,-53.01,74.0,2018
4,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2022,0.14,0.57,0.08,-1.14,4.56,27.69,-31.53,1642.0,2018
5,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2023,0.11,0.66,0.07,-0.48,4.35,41.81,-20.22,1795.0,2018
6,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2024,0.14,0.68,0.09,-0.41,6.19,39.93,-16.45,1917.0,2018
7,14338,2018-01-31,0.072502,85.65,376368.0,319951.0,2018,319951000.0,27403803150.0,14338.0,...,2025,0.12,0.72,0.09,-0.27,6.17,46.53,-12.58,2028.0,2018
8,14338,2018-02-28,-0.056743,80.79,561565.0,316118.0,2018,316118000.0,25539173220.000004,14338.0,...,2018,0.09,0.64,0.05,1.39,2.53,28.38,39.35,1183.0,2018
9,14338,2018-02-28,-0.056743,80.79,561565.0,316118.0,2018,316118000.0,25539173220.000004,14338.0,...,2019,0.09,0.63,0.06,-1.83,3.07,36.13,-66.04,1303.0,2018


In [20]:
# Sort by company and trade_date for time-series alignment
merged_clean = merged_clean.sort_values(["company_name", "trade_date"]).reset_index(drop=True)
print("Dataset sorted by company and trade date.")

Dataset sorted by company and trade date.


In [21]:
# Number of duplicate rows
num_duplicates = merged_clean.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

# Optionally, drop duplicates
merged_clean = merged_clean.drop_duplicates().reset_index(drop=True)
print("Duplicates removed, dataset now has rows:", merged_clean.shape[0])

Number of duplicate rows: 0
Duplicates removed, dataset now has rows: 2688


In [22]:
# --- Save the final dataset ---
merged_clean.to_csv("top4_hotels_final_2018_2024.csv", index=False)
merged_clean.to_parquet("top4_hotels_final_2018_2024.parquet", index=False)

print("Final dataset saved successfully!")
print(f"Rows, Columns: {merged_clean.shape}")

Final dataset saved successfully!
Rows, Columns: (2688, 33)


In [23]:
import pandas as pd
import numpy as np

# --- 1. Convert monthly stock data to yearly returns ---
# Assume merged_clean has 'trade_date', 'return', 'company_name', 'ticker', 'year_crsp'
merged_clean['year_crsp'] = merged_clean['trade_date'].dt.year

# Group by company and year, calculate cumulative annual return
annual_stock = merged_clean.groupby(['company_name', 'ticker', 'year_crsp']).apply(
    lambda x: (np.prod(1 + x['return'].fillna(0)) - 1)
).reset_index(name='annual_return')

# --- 2. Aggregate financial metrics by year (year-end) ---
# Assume 'year_comp' exists as the fiscal year
financial_yearly = merged_clean.groupby(['company_name', 'ticker', 'year_comp']).agg({
    'ROA': 'last',
    'ROE': 'last',
    'PE_ratio': 'last',
    'PB_ratio': 'last',
    'EPS': 'last',
    'FCF': 'last',
    'profit_margin': 'last',
    'asset_turnover': 'last',
    'market_cap': 'last'
}).reset_index()

# --- 3. Merge stock returns with financials ---
annual_data = pd.merge(
    financial_yearly,
    annual_stock,
    left_on=['company_name', 'ticker', 'year_comp'],
    right_on=['company_name', 'ticker', 'year_crsp'],
    how='left'
)

# Optional: drop duplicate year columns
annual_data = annual_data.drop(columns=['year_crsp'])

# --- 4. Round values for readability ---
metrics_to_round = ['ROA', 'ROE', 'PE_ratio', 'PB_ratio', 'EPS', 'FCF', 'profit_margin', 'asset_turnover', 'market_cap', 'annual_return']
annual_data[metrics_to_round] = annual_data[metrics_to_round].round(2)

# --- 5. Quick check ---
print("Rows, Columns:", annual_data.shape)
print("Preview first 10 rows:")
annual_data.head(10)

Rows, Columns: (32, 13)
Preview first 10 rows:


/var/folders/sb/mh5ps_d913v_czrnc8sp4n500000gn/T/ipykernel_23248/1859657626.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  annual_stock = merged_clean.groupby(['company_name', 'ticker', 'year_crsp']).apply(


,company_name,ticker,year_comp,ROA,ROE,PE_ratio,PB_ratio,EPS,FCF,profit_margin,asset_turnover,market_cap,annual_return
0,HILTON WORLDWIDE HOLDINGS,HLT,2018,0.05,1.39,28.38,39.35,2.53,1183.0,0.09,0.64,60252664800.0,-0.55
1,HILTON WORLDWIDE HOLDINGS,HLT,2019,0.06,-1.83,36.13,-66.04,3.07,1303.0,0.09,0.63,60252664800.0,33.16
2,HILTON WORLDWIDE HOLDINGS,HLT,2020,-0.04,0.48,-43.1,-20.68,-2.58,662.0,-0.17,0.26,60252664800.0,0.04
3,HILTON WORLDWIDE HOLDINGS,HLT,2021,0.03,-0.5,106.15,-53.01,1.47,74.0,0.07,0.37,60252664800.0,13.93
4,HILTON WORLDWIDE HOLDINGS,HLT,2022,0.08,-1.14,27.69,-31.53,4.56,1642.0,0.14,0.57,60252664800.0,-0.81
5,HILTON WORLDWIDE HOLDINGS,HLT,2023,0.07,-0.48,41.81,-20.22,4.35,1795.0,0.11,0.66,60252664800.0,18.21
6,HILTON WORLDWIDE HOLDINGS,HLT,2024,0.09,-0.41,39.93,-16.45,6.19,1917.0,0.14,0.68,60252664800.0,10.78
7,HILTON WORLDWIDE HOLDINGS,HLT,2025,0.09,-0.27,46.53,-12.58,6.17,2028.0,0.12,0.72,60252664800.0,NaN
8,HYATT HOTELS CORP,H,2018,0.1,0.21,9.96,2.09,6.79,44.0,0.17,0.58,6638684200.0,-0.46
9,HYATT HOTELS CORP,H,2019,0.09,0.19,12.25,2.37,7.32,27.0,0.15,0.6,6638684200.0,9.44


In [26]:
# --- 6. Flag missing stock data ---
annual_data['stock_data_missing'] = annual_data['annual_return'].isna()

In [27]:
# --- 7. Save final annual dataset ---
print("Saving the final annual dataset...")

# Save as CSV
annual_data.to_csv("top4_hotels_annual_2018_2025.csv", index=False)

# Save as Parquet (faster to load in the app)
annual_data.to_parquet("top4_hotels_annual_2018_2025.parquet", index=False)

print("Final dataset saved successfully!")

Saving the final annual dataset...
Final dataset saved successfully!


In [ ]:
db.close()